# 🚀 Image Preprocessing Pipeline Demo

This notebook demonstrates how to:
1. Process individual images with coordinates
2. Batch process all images
3. Load preprocessed images for model training

## 📁 Folder Structure
```
data/
├── train/              # Original images
├── train_labels.csv    # Image coordinates
└── processed/          # NEW! Processed data
    ├── images/         # Preprocessed .npy files
    └── metadata/       # Metadata CSV
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from helpers import (
    process_and_save_image,
    batch_process_and_save_images,
    load_processed_images,
    load_image
)

## 1️⃣ Process a Single Image (Example)

Let's process one image to understand the pipeline

In [ ]:
# Load the labels to get coordinates
labels_df = pd.read_csv('data/train_labels.csv')
print(f"Total images in dataset: {len(labels_df)}")
print("\nFirst 5 rows:")
labels_df.head()

In [ ]:
# Process a single image
sample = labels_df.iloc[0]
image_id = sample['filename'].replace('.jpg', '')
coords = (sample['x'], sample['y'])

print(f"Processing Image ID: {image_id}")
print(f"Original coordinates: {coords}")

metadata = process_and_save_image(
    image_id=image_id,
    image_path=f"data/train/{sample['filename']}",
    coords=coords,
    output_folder="data/processed/images",
    target_size=(224, 224),
    normalize=True
)

print("\n✅ Processing complete!")
print(f"Metadata: {metadata}")

In [ ]:
# Visualize: Original vs Processed
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Load original image
original = load_image(f"data/train/{sample['filename']}")
axes[0].imshow(original)
axes[0].plot(coords[0], coords[1], 'r+', markersize=15, markeredgewidth=2)
axes[0].set_title(f"Original Image ({original.shape[1]}x{original.shape[0]})")
axes[0].axis('off')

# Load processed image
processed = np.load(metadata['save_path'])
# Denormalize for visualization
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])
processed_vis = (processed * std + mean).clip(0, 1)

axes[1].imshow(processed_vis)
scaled_coords = metadata['scaled_coords']
axes[1].plot(scaled_coords[0], scaled_coords[1], 'r+', markersize=15, markeredgewidth=2)
axes[1].set_title(f"Processed Image ({processed.shape[1]}x{processed.shape[0]})")
axes[1].axis('off')

plt.tight_layout()
plt.show()

print(f"Original coordinates: {coords}")
print(f"Scaled coordinates: {scaled_coords}")

## 2️⃣ Batch Process All Images

Process all images at once and save to the processed folder

In [ ]:
# Batch process all images
# WARNING: This will process ALL images in the dataset (may take time!)
# For testing, you can create a smaller CSV with fewer images first

metadata_df = batch_process_and_save_images(
    csv_file='data/train_labels.csv',
    image_folder='data/train',
    output_folder='data/processed/images',
    metadata_file='data/processed/metadata/metadata.csv',
    target_size=(224, 224),
    normalize=True,
    mean=[0.485, 0.456, 0.406],  # ImageNet mean
    std=[0.229, 0.224, 0.225]    # ImageNet std
)

In [ ]:
# Check the metadata
print(f"Total processed images: {len(metadata_df)}")
print("\nMetadata columns:", metadata_df.columns.tolist())
print("\nFirst 5 rows:")
metadata_df.head()

## 3️⃣ Load Preprocessed Images for Model Training

Now you can directly load preprocessed images for training!

In [ ]:
# Load all preprocessed images (or specify max_images for a subset)
images, coordinates, metadata = load_processed_images(
    metadata_file='data/processed/metadata/metadata.csv',
    max_images=100  # Load first 100 images for demo
)

print(f"\n📊 Data ready for training!")
print(f"Images shape: {images.shape}")
print(f"Coordinates shape: {coordinates.shape}")
print(f"\nSample coordinates (first 5):")
print(coordinates[:5])

In [ ]:
# Visualize some processed images
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

for i in range(8):
    # Denormalize for visualization
    img = (images[i] * std + mean).clip(0, 1)
    
    axes[i].imshow(img)
    axes[i].plot(coordinates[i][0], coordinates[i][1], 'r+', markersize=15, markeredgewidth=2)
    axes[i].set_title(f"Image {i} - Coords: {coordinates[i]}")
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 4️⃣ Ready for Model Training!

### Simple Model Training Example

In [ ]:
# Example: Split data for training
from sklearn.model_selection import train_test_split

# Load all data
X, y, _ = load_processed_images('data/processed/metadata/metadata.csv')

# Split into train and validation
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} images")
print(f"Validation set: {X_val.shape[0]} images")
print(f"\nImage shape: {X_train.shape[1:]}")
print(f"Coordinate shape: {y_train.shape[1:]}")

print("\n✅ Data ready to feed directly into your model!")

## 📊 Summary

### What we accomplished:
1. ✅ Created `data/processed/` folder structure
2. ✅ Processed images with resizing (224x224)
3. ✅ Normalized images using ImageNet statistics
4. ✅ Scaled coordinates proportionally
5. ✅ Saved as .npy files for fast loading
6. ✅ Generated metadata CSV for tracking

### Advantages:
- 🚀 **Fast loading**: No need to process images every time
- 💾 **Efficient storage**: NumPy binary format
- 🎯 **Ready for training**: Direct model input
- 📊 **Tracked metadata**: All information preserved

### Usage in your training pipeline:
```python
from helpers import load_processed_images

# Load preprocessed data
images, coordinates, metadata = load_processed_images()

# Feed directly to model
model.fit(images, coordinates)
```